In [1]:
import pandas as pd
import plotly.graph_objects as go
import glob
import numpy as np
import os
import plotly.colors
import re

def to_rgba_string(color_str, alpha=0.6):
    """
    Converts a color string (hex, rgb(...), or name) to a
    Plotly-compatible rgba string with a specified alpha.
    """
    # This function is designed to be robust to different color formats.
    if isinstance(color_str, str) and color_str.startswith('rgb'):
        # Handle the 'rgb(r, g, b)' format directly.
        return color_str.replace('rgb', 'rgba').replace(')', f', {alpha})')
    
    # For hex strings and color names, use Plotly's own robust converters.
    try:
        from matplotlib.colors import to_rgb
        rgb_tuple = tuple(int(c * 255) for c in to_rgb(color_str))
    except (ValueError, ImportError):
        # Fallback for any other unexpected color format
        try:
            rgb_tuple = plotly.colors.hex_to_rgb(color_str)
        except ValueError:
            return f"rgba(204, 204, 204, {alpha})" # Default to grey
            
    return f"rgba({rgb_tuple[0]}, {rgb_tuple[1]}, {rgb_tuple[2]}, {alpha})"

# --- 1. Calculate Source Data Sizes ---
sources = {
    "ACL": "**/acl_results.csv", "ACM": "**/*_acm*.csv", "DBLP": "**/*_dblp.csv",
    "OpenAlex": "**/*_openalex.csv", "Scopus": "**/*_scopus.csv", "IEEE": "**/*_ieee.csv"
}
source_counts = {}
for source, pattern in sources.items():
    count = 0
    files = glob.glob(pattern, recursive=True)
    for f in files:
        try:
            df = pd.read_csv(f)
            count += len(df)
        except Exception as e:
            print(f"Could not process file {f}: {e}")
    source_counts[source] = count

total_raw_count = sum(source_counts.values())

# --- 2. Load Final Dataset and Calculate Flows ---
final_df = pd.read_csv('final_taxonomy.csv')
deduplicated_count = len(final_df)
excluded_df = final_df[final_df['included in review'].astype(str).str.startswith('X')].copy()

# --- Map exclusion reasons to new labels ---
exclusion_label_map = {
    "X (P)": "Prompt Methods", "X (RAG)": "RAG Methods",
    "X (PR)": "Not Peer-Reviewed", "X (NR)": "Not Relevant"
}
excluded_df['included in review'] = excluded_df['included in review'].map(exclusion_label_map).fillna(excluded_df['included in review'])
exclusion_counts = excluded_df['included in review'].value_counts().to_dict()

# --- Sort exclusion reasons for consistent ordering ---
sorted_exclusion_reasons = sorted(exclusion_counts.keys())

included_df = final_df[final_df['included in review'].isin(['Y', 'Survey'])]
included_count = len(included_df)

In [2]:

# --- 3. Prepare Data for Sankey Diagram ---
def style_label(label, align='center'):
    # Aligns text as desired
    text_align = 'left' if align == 'left' else 'center'
    return f'<span style="font-family: Questrial, sans-serif; font-weight: bold; text-align:{text_align}; background-color: white; border: 1px solid black; padding: 3px; border-radius: 4px; display:inline-block;">{label}</span>'

# --- Add a new node for "Duplicates" ---
node_labels_raw = {
    **{source: f"{source} ({count})" for source, count in source_counts.items()},
    "Combined": f"Combined Papers ({total_raw_count})",
    "Duplicates": f"Duplicate Papers ({total_raw_count-deduplicated_count})",
    "DeduplicatedDataset": f"De-duplicated Papers ({deduplicated_count})",
    **{reason: f"{reason} ({exclusion_counts[reason]})" for reason in sorted_exclusion_reasons},
    "Included": f"Included in Review ({included_count})"
}

# --- Left-align labels for left-most nodes ---
labels = []
for name, label_text in node_labels_raw.items():
    if name in source_counts.keys():  # leftmost sources
        labels.append(style_label(label_text, align='left'))  # <<< CHANGE
    else:
        labels.append(style_label(label_text))

label_indices = {name: i for i, name in enumerate(node_labels_raw.keys())}

source_nodes, target_nodes, values = [], [], []

# --- Define Flows ---
combined_idx = label_indices["Combined"]
deduplicated_idx = label_indices["DeduplicatedDataset"]
duplicates_idx = label_indices["Duplicates"]  # <<< CHANGE

# Flow: individual sources → Combined
for source in source_counts.keys():
    source_nodes.append(label_indices[source])
    target_nodes.append(combined_idx)
    values.append(source_counts[source])

# Flow: Combined → Duplicates
duplicates_value = total_raw_count - deduplicated_count  # <<< CHANGE
source_nodes.append(combined_idx)
target_nodes.append(duplicates_idx)
values.append(duplicates_value)

# Flow: Combined → De-duplicated Dataset
source_nodes.append(combined_idx)
target_nodes.append(deduplicated_idx)
values.append(deduplicated_count)

# Flow: De-duplicated Dataset → Exclusion Reasons
for reason in sorted_exclusion_reasons:
    source_nodes.append(deduplicated_idx)
    target_nodes.append(label_indices[reason])
    values.append(exclusion_counts[reason])

# Flow: De-duplicated Dataset → Included
source_nodes.append(deduplicated_idx)
target_nodes.append(label_indices["Included"])
values.append(included_count)

# --- 4. Define Node Positions and Colors ---
num_sources = len(source_counts)
num_exclusions = len(sorted_exclusion_reasons)

# Slightly expanded positions for new node
node_x = (
    [0.01] * num_sources
    + [0.25, 0.4, 0.6]  # Combined, Duplicates, Deduplicated
    + [0.8] * num_exclusions
    + [1.0]
)

# Set Y coordinates manually for clarity
node_y = (
    list(np.linspace(0.01, 1, num_sources))
    + [0.5, 0.3, 0.7]  # Combined, Duplicates, De-duplicated
    + list(np.linspace(0.05, 0.9, num_exclusions)-0.3)
    + [1]  # <<< CHANGE: Included node moved to bottom
)

# --- Node Color Palette ---
color_palette = plotly.colors.qualitative.Pastel[:num_sources]
color_palette += ['#FFC107', '#F44336', '#FF9800']  # Combined, Duplicates, Deduplicated
color_palette += ['#F44336'] * num_exclusions
color_palette += ['#4CAF50']

# --- Link Color Palette ---
link_colors = []
included_idx = label_indices["Included"]
combined_idx = label_indices["Combined"]
exclusion_indices = [label_indices[reason] for reason in sorted_exclusion_reasons]

for source_node_idx, target_node_idx in zip(source_nodes, target_nodes):
    if target_node_idx == combined_idx:
        link_colors.append(to_rgba_string(color_palette[source_node_idx]))
    elif target_node_idx == duplicates_idx:
        link_colors.append(to_rgba_string('red'))
    elif source_node_idx == combined_idx:
        link_colors.append(to_rgba_string(color_palette[target_node_idx]))
    elif target_node_idx == included_idx:
        link_colors.append(to_rgba_string('green'))
    elif target_node_idx in exclusion_indices:
        link_colors.append(to_rgba_string('red'))
    else:
        link_colors.append(to_rgba_string('lightgray'))

# --- 5. Create and Display the Sankey Diagram ---
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=25, thickness=20, line=dict(color="black", width=0.6),
        label=labels, x=node_x, y=node_y, color=color_palette
    ),
    link=dict(source=source_nodes, target=target_nodes, value=values, color=link_colors)
)])

fig.update_layout(
    title_text="Flow of Literature Review Papers",
    font=dict(size=14, family="Questrial, sans-serif"),
    margin=dict(l=50, r=50, t=50+10, b=50)
)

fig.write_image("sankey_diagram_dups.png", width=900, height=400, scale=3)
print("Sankey diagram saved as 'sankey_diagram_dups.png'")
fig.show()


Sankey diagram saved as 'sankey_diagram_dups.png'
